# Diagnóstico Bronze — steamcharts_2025_history_data

Tabla analizada: `steam_game_intelligence.bronze.steamcharts_2025_history_data`. Se revisan cobertura temporal, parsing del campo `month` (string), duplicados mensuales y métricas de jugadores. No se transforma todavía el campo temporal.

> Ejecuta las celdas en orden. Estas consultas son de diagnóstico: no modifican las tablas Bronze.

## 1. Volumen y cobertura temporal

In [ ]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT steam_appid) AS distinct_app_ids,
  COUNT(DISTINCT month) AS distinct_month_strings,
  MIN(month) AS min_month_string,
  MAX(month) AS max_month_string,
  MIN(_ingested_at) AS first_ingested_at,
  MAX(_ingested_at) AS last_ingested_at
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data;

## 2. Nulos por columna

In [ ]:
%sql
SELECT column_name, null_count
FROM (
  SELECT
    COUNT_IF(month IS NULL) AS month,
    COUNT_IF(avg_players IS NULL) AS avg_players,
    COUNT_IF(gain IS NULL) AS gain,
    COUNT_IF(gain_percent IS NULL) AS gain_percent,
    COUNT_IF(peak_players IS NULL) AS peak_players,
    COUNT_IF(name IS NULL) AS name,
    COUNT_IF(steam_appid IS NULL) AS steam_appid,
    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,
    COUNT_IF(_source IS NULL) AS _source
  FROM steam_game_intelligence.bronze.steamcharts_2025_history_data
) UNPIVOT (null_count FOR column_name IN (
  month, avg_players, gain, gain_percent, peak_players, name,
  steam_appid, _ingested_at, _source
))
WHERE null_count > 0
ORDER BY null_count DESC;

## 3. Formato de month
Se prueba el formato `yyyy-MM`. Si el dataset usa otro formato, los resultados nulos indicarán que hay que adaptar el parser antes de limpiar.

In [ ]:
%sql
SELECT month,
       TRY_TO_DATE(month, 'yyyy-MM') AS parsed_month,
       COUNT(*) AS row_count
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data
GROUP BY month, TRY_TO_DATE(month, 'yyyy-MM')
ORDER BY parsed_month NULLS FIRST, month
LIMIT 200;

In [ ]:
%sql
SELECT
  COUNT_IF(month IS NOT NULL AND TRY_TO_DATE(month, 'yyyy-MM') IS NULL) AS unparsed_month_rows,
  COUNT_IF(month IS NOT NULL AND TRIM(month) = '') AS blank_month_rows,
  COUNT_IF(steam_appid <= 0) AS invalid_appid_rows,
  COUNT_IF(avg_players < 0) AS negative_avg_players_rows,
  COUNT_IF(peak_players < 0) AS negative_peak_players_rows,
  COUNT_IF(gain_percent < -100) AS gain_percent_below_minus_100_rows,
  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data;

## 4. Duplicados por juego y mes

In [ ]:
%sql
SELECT steam_appid, month, COUNT(*) AS row_count,
       COUNT(DISTINCT name) AS distinct_names,
       COUNT(DISTINCT avg_players) AS distinct_avg_players,
       COUNT(DISTINCT peak_players) AS distinct_peak_players
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data
GROUP BY steam_appid, month
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 200;

## 5. Identificadores o nombres inconsistentes

In [ ]:
%sql
SELECT steam_appid,
       COUNT(DISTINCT name) AS distinct_names,
       MIN(name) AS example_name,
       COUNT(*) AS rows_count
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data
WHERE steam_appid IS NOT NULL
GROUP BY steam_appid
HAVING COUNT(DISTINCT name) > 1
ORDER BY rows_count DESC
LIMIT 100;

## 6. Métricas fuera de rango o sospechosas

In [ ]:
%sql
SELECT *
FROM steam_game_intelligence.bronze.steamcharts_2025_history_data
WHERE steam_appid <= 0
   OR (avg_players < 0)
   OR (peak_players < 0)
   OR (avg_players IS NOT NULL AND peak_players IS NOT NULL AND avg_players > peak_players)
   OR (gain_percent < -100)
   OR (month IS NOT NULL AND TRY_TO_DATE(month, 'yyyy-MM') IS NULL)
   OR name IS NULL OR TRIM(name) = ''
LIMIT 200;